In [36]:
# Data manipulation
import pandas as pd
import numpy as np


# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

# math library
import math
import statistics

# warning management library
import warnings

# DOS like library
import os

# Date management
from datetime import *
from dateutil.relativedelta import *
from dateutil.parser import *


# Images analysis
from PIL import Image
from PIL import ImageOps

# Text Mining
import nltk
nltk.download('punkt')
from wordcloud import WordCloud
from collections import Counter
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt_tab')

# Machine learning
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [37]:
warnings.filterwarnings('ignore')
sns.set()
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

### Loading of file title.basics.tz

In [38]:
titles = pd.read_csv("../Raw_Data/title.basics.tsv", sep='\t', nrows = 10000)

In [39]:
titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   tconst          10000 non-null  object
 1   titleType       10000 non-null  object
 2   primaryTitle    10000 non-null  object
 3   originalTitle   10000 non-null  object
 4   isAdult         10000 non-null  int64 
 5   startYear       10000 non-null  object
 6   endYear         10000 non-null  object
 7   runtimeMinutes  10000 non-null  object
 8   genres          10000 non-null  object
dtypes: int64(1), object(8)
memory usage: 703.3+ KB


In [40]:
titles.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [41]:
titles_reduced = titles[["tconst", "primaryTitle", "genres"]]

In [42]:
titles_reduced.head()

,tconst,primaryTitle,genres
0,tt0000001,Carmencita,"Documentary,Short"
1,tt0000002,Le clown et ses chiens,"Animation,Short"
2,tt0000003,Poor Pierrot,"Animation,Comedy,Romance"
3,tt0000004,Un bon bock,"Animation,Short"
4,tt0000005,Blacksmith Scene,Short


In [43]:
titles_reduced.rename(columns={"primaryTitle": "title"}, inplace=True)
titles_reduced["genres"] = titles_reduced["genres"].apply(lambda x: x.replace(",", " "))
titles_reduced.head()

,tconst,title,genres
0,tt0000001,Carmencita,Documentary Short
1,tt0000002,Le clown et ses chiens,Animation Short
2,tt0000003,Poor Pierrot,Animation Comedy Romance
3,tt0000004,Un bon bock,Animation Short
4,tt0000005,Blacksmith Scene,Short


### 1. Starting Natural Language Processing

#### 1.1 Lemmatization and Tokenization

In [44]:
# list of french stopwords : https://www.ranks.nl/stopwords/french
french_stopwords = ["alors","au","aucun","aussi","autre","avant","avec","avoir","bon","car","ce","cela","ces","ceux","chaque","ci",
                    "comme","comment","dans","des","du",
"dedans","dehors","depuis","devrait","doit","donc","début","elle","elles","en","encore","essai","est","et","eu","fait","faites",
"fois","font","hors","ici","il","ils","je","juste",
"la","le","les","leur","là","ma","maintenant","mais","mes","mien","moins","mon","mot","même","ni","notre","nous","ou","où","par",
"parce","pas","peut","peu","plupart","pour","pourquoi",
"quand","que","quel","quelle","quelles","quels","qui","sa","sans","ses","seulement","si","sien","son","sont","sous","soyez","sur",
"ta","tandis","tellement","tels","tes","ton","tous",
"tout","trop","très","tu","voient","vont","votre","vous","vu","ça","étaient","état","étions","été","être", "un"]

In [45]:
# creating a list of stopwords for a specific text : with text most common words, and with french end english typical stopwords list
def stopwords_list(text,sw_nb):
    # Lemmatization
    text=text.lower()
    lemmatizer = WordNetLemmatizer()
    text=lemmatizer.lemmatize(text)
    
    # Tokenization
    words_list=nltk.word_tokenize(text)
    words_list= [word for word in words_list if word.isalnum()]
    
    # Suppress digits
    words_list=[w for w in words_list if w.isalpha()]
    
    # wordcount and stopwords
    word_counts = Counter(words_list)
    stopwords_list=[count[0] for count in word_counts.most_common(sw_nb)]
    
    sw = set()
    sw.update(stopwords_list)
    sw.update(tuple(nltk.corpus.stopwords.words('english')))
    sw=list(sw)
    # adding french stopwords
    for fsw in french_stopwords:
        sw.append(fsw)
    return sw

In [46]:
# Text lemmatization and tokenization
def tokenize_text(text):
    # Lemmatization
    text=text.lower()
    lemmatizer = WordNetLemmatizer()
    text=lemmatizer.lemmatize(text)
    
    # Tokenization
    words_list=nltk.word_tokenize(text)
    words_list= [word for word in words_list if word.isalnum()]
    
    # Suppress digits
    words_list=[lemmatizer.lemmatize(w) for w in words_list if w.isalpha()]
    return words_list

In [47]:
# definition of a wordcloud function to have a small visualisation
def word_cloud(words_list):
    wordcloud=WordCloud()
    sentence=' '.join(words_list)
    wordcloud.generate(sentence)
    # create a figure
    fig, ax = plt.subplots(1,1, figsize = (9,6))
    # add interpolation = bilinear to smooth things out
    plt.imshow(wordcloud, interpolation='bilinear')
    # and remove the axis
    plt.axis("off")

#### 1.2 Text pre-processing

In [48]:
text = (titles_reduced.iloc[:,1] + " " + titles_reduced.iloc[:,2] + " ").sum()

In [49]:
text

"Carmencita Documentary Short Le clown et ses chiens Animation Short Poor Pierrot Animation Comedy Romance Un bon bock Animation Short Blacksmith Scene Short Chinese Opium Den Short Corbett and Courtney Before the Kinetograph Short Sport Edison Kinetoscopic Record of a Sneeze Documentary Short Miss Jerry Romance Leaving the Factory Documentary Short Akrobatisches Potpourri Documentary Short The Arrival of a Train Documentary Short The Photographical Congress Arrives in Lyon Documentary Short The Waterer Watered Comedy Short Around a Cabin Animation Comedy Short Boat Leaving the Port Documentary Short Italienischer Bauerntanz Documentary Short Das boxende Känguruh Short Sport The Clown Barber Comedy Short The Derby 1895 Documentary Short Sport Blacksmith Scene Documentary Short The Sea Documentary Short Opening of the Kiel Canal News Short The Oxford and Cambridge University Boat Race News Short Sport The Messers. Lumière at Cards Documentary Short Cordeliers' Square in Lyon Documentary

In [50]:
sw=stopwords_list(text,10)

In [51]:
sw[:10]

['ourselves',
 'and',
 'herself',
 "they're",
 'did',
 "we'll",
 'your',
 'drama',
 'will',
 "we've"]

In [52]:
words_list = tokenize_text(text)

In [53]:
words_list_wo_sw=[w for w in words_list if w not in sw]

In [54]:
words_list_wo_sw[:20]

['carmencita',
 'documentary',
 'clown',
 'chiens',
 'poor',
 'pierrot',
 'bock',
 'blacksmith',
 'scene',
 'chinese',
 'opium',
 'den',
 'corbett',
 'courtney',
 'kinetograph',
 'sport',
 'edison',
 'kinetoscopic',
 'record',
 'sneeze']